# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'
# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access the metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their IDs, and print the field and column IDs for each.

In [ ]:
# List all record sets and their fields and columns, referencing by @id
record_sets = dataset.list_record_sets()
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs['name']}")
    if 'fields' in rs:
        print("  Fields (@id):")
        for f in rs['fields']:
            print(f"    - @id: {f['@id']} | name: {f.get('name', '')}")
    if 'columns' in rs:
        print("  Columns (@id):")
        for c in rs['columns']:
            print(f"    - @id: {c['@id']} | name: {c.get('name', '')}")
    print()
# Optionally, preview a few records from each record set
for rs in record_sets:
    print(f"First record for record set {rs['@id']}:")
    records = dataset.records(record_set=rs['@id'])
    for rec in records:
        print(json.dumps(rec, indent=2))
        break  # Only print first record
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s discovered above.

In [ ]:
# Collect record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load records for each record set ID
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"DataFrame columns for record set @id {record_set_id}:")
        print(dataframes[record_set_id].columns.tolist())
        print(dataframes[record_set_id].head(3))
    else:
        print(f"No records found for record set {record_set_id}.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing numeric fields, grouping by key attributes, and more.

For demonstration, select a numeric field and perform basic filtering and normalization.

In [ ]:
# EDA for a specific record set

# Pick the first record set with actual records
main_record_set_id = next(iter(dataframes))
df = dataframes[main_record_set_id]

# Identify numeric fields (columns containing numeric values)
numeric_fields = df.select_dtypes(include=['int', 'float']).columns.tolist()
if not numeric_fields:
    # fallback: try PHQ-9, GAD-7, or PSQ fields by @id
    # Let's print columns for user reference
    print("No numeric fields detected automatically. Showing all columns:")
    print(df.columns.tolist())
    # User might want to pick e.g. 'phq9_score' or 'gad7_score' depending on schema
else:
    # Use first numeric field
    numeric_field_id = numeric_fields[0]

    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field if available
    group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if group_fields:
        group_field_id = group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# For the main record set and numeric field (from above), plot histogram

if 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id} (@id)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If a grouping field exists, visualize mean scores by group
    if 'group_field_id' in locals():
        plt.figure(figsize=(10,5))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
        plt.title(f'Mean {numeric_field_id} by {group_field_id} (@id)')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrates how to load, review, and analyze a Croissant AI-Ready dataset using `mlcroissant`, referencing all entities by their `@id`.

**Key Findings:**
- The dataset contains survey records with demographic and mental health assessment fields (e.g., PHQ-9, GAD-7, PSQ).
- Numeric fields (such as assessment scores) can be filtered and normalized for further analysis.
- Grouped statistics and visualizations reveal trends and group differences.

Further analysis can be extended by exploring additional record sets or fields referenced by their `@id`.